<a href="https://colab.research.google.com/github/zaima-sohail/flyrank-ml-internship/blob/main/Copy_of_w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zaima-sohail/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

## 1. My lane

**Lane 2: Refresh / Content Opportunity Scoring**

Question: Which pages should be reviewed first for refresh, expansion, protection,
pruning, or monitoring?

I'm picking this lane because it directly extends what I already found in Weeks 1-2:
`trend_direction`, `days_since_last_update`, and `impressions_90d` already showed
measurable patterns (a hand rule scored Precision@50 of 0.68, and a simple depth-2
tree beat it deeper into the ranking at 0.68 vs 0.58 on held-out data). This lane
lets me build directly on that foundation rather than starting from zero, and it
matches the lane guide's own baseline formula almost exactly, so I have a clear,
established starting point to test against.

I can confirm or change this by end of Week 4.

## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

## 2. The question

**Search question:** Given a page's observable signals (staleness, visibility,
position, CTR, engagement), can we produce a ranked review queue that surfaces
the pages most worth reviewing first — for refresh, protection, or monitoring?

**Unit of analysis:** one content page (`content_id`), scored at one point in time.

**The decision this improves:** out of thousands of pages, which small number
(e.g. top 50) a content team reviews and acts on this cycle, given limited
editorial capacity.

**The action someone takes:** an SEO/content editor pulls the ranked queue, reads
the reason codes attached to each page (e.g. `stale_visible_page`,
`declining_with_demand`, `low_ctr_visible_page`), and decides refresh, expand,
protect, prune, or monitor for each one.

**Cost of a wrong recommendation:**
- False positive (page flagged but didn't need review): wasted editorial hours
  on a page that was actually fine.
- False negative (a genuinely declining or high-opportunity page never surfaces
  in the queue): real traffic loss goes unaddressed, and by the time it's
  noticed elsewhere, the gap has widened.
- Given limited review capacity, false positives cost time directly, while false
  negatives cost missed opportunity — I'll treat both as real costs and use
  precision@K (matching actual review capacity) rather than generic accuracy,
  per the lane guide's guidance in section 11.

**Why data/ML helps here, not just manual review:** with tens of thousands of
pages (and up to ~520K in the full warehouse `dim_content`), no team can review
every page every cycle. A ranked, evidence-backed queue turns an unmanageable
list into a short, defensible priority order — exactly what Notebook 2
demonstrated when the tree beat the hand rule deeper into the ranking.

## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [ ]:
! git clone https://github.com/zaima-sohail/flyrank-ml-internship.git

fatal: destination path 'flyrank-ml-internship' already exists and is not an empty directory.


In [ ]:
import pandas as pd

df = pd.read_csv("flyrank-ml-internship/data/raw/content_refresh_anonymized.csv")

print(df.head())
print(df.shape)

             content_id          client_id  search_volume  competition  \
0  content_304f48230142  client_f369cb89fc           10.0         0.67   
1  content_a1fb4e703a9e  client_4e07408562           90.0         0.01   
2  content_9aa793d4d895  client_7f2253d7e2            0.0         0.00   
3  content_331d6c4de07b  client_19581e27de           10.0         0.00   
4  content_d99b7a2d90ca  client_3fdba35f04            0.0         0.00   

  competition_level   cpc     content_type    main_intent  word_count  \
0              HIGH  2.05  keyword article  transactional      3221.0   
1               LOW  0.05  keyword article  informational      2481.0   
2               LOW  0.00  keyword article  informational      3515.0   
3               LOW  0.00  keyword article     commercial         NaN   
4               LOW  0.00  keyword article  informational      2803.0   

   char_count  ... char_count_tier   ctr  avg_position  engagement_rate  \
0     20457.0  ...     15000-25000  0.76 

In [ ]:
print(df.describe(include="all"))

                  content_id          client_id  search_volume   competition  \
count                  30000              30000   27532.000000  27532.000000   
unique                 30000                 32            NaN           NaN   
top     content_6880eb215048  client_19581e27de            NaN           NaN   
freq                       1               7008            NaN           NaN   
mean                     NaN                NaN     158.882391      0.146954   
std                      NaN                NaN    1518.270825      0.285241   
min                      NaN                NaN       0.000000      0.000000   
25%                      NaN                NaN       0.000000      0.000000   
50%                      NaN                NaN      10.000000      0.000000   
75%                      NaN                NaN      20.000000      0.130000   
max                      NaN                NaN   74000.000000      1.000000   

       competition_level           cpc 

In [ ]:
print(df.isnull().sum())

content_id                    0
client_id                     0
search_volume              2468
competition                2468
competition_level          2610
cpc                        2468
content_type                  0
main_intent                2374
word_count                 7699
char_count                 7699
provider_used             21438
model_used                 5733
impressions_90d               0
clicks_90d                    0
pageviews_90d                 0
sessions_90d                  0
users_90d                     0
engaged_sessions_90d          0
ai_sessions_90d               0
scroll_events_90d             0
days_with_impressions         0
days_with_sessions            0
impressions_last_30d          0
clicks_last_30d               0
sessions_last_30d             0
impressions_prev_30d          0
clicks_prev_30d               0
sessions_prev_30d             0
content_age_days              0
age_tier                      0
age_tier_order                0
days_sin

In [ ]:
print("Rows:", len(df))
print("Columns:", len(df.columns))

Rows: 30000
Columns: 44


In [ ]:
print(df.columns.tolist())

['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [ ]:
import pandas as pd

df = pd.read_csv("flyrank-ml-internship/data/raw/content_refresh_anonymized.csv")

print("Rows:", len(df))
print("Columns:", len(df.columns))

print("Average CTR:", round(df["ctr"].mean(), 2))
print("Average Position:", round(df["avg_position"].mean(), 2))
print("Average Content Age:", round(df["content_age_days"].mean(), 0))

print("\nTrend Direction Counts:")
print(df["trend_direction"].value_counts())

Rows: 30000
Columns: 44
Average CTR: 0.51
Average Position: 16.34
Average Content Age: 256.0

Trend Direction Counts:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64


# Quick Look at the Data

The starter dataset contains **30,000 content pages** and **44 features**, providing enough observations for meaningful analysis.

Some initial observations include:

- Average click-through rate (CTR): **0.51**
- Average search position: **16.34**
- Average content age: **256 days**

The trend distribution also suggests many pages may benefit from attention:

- Down: **16,262 pages**
- Stable: **5,962 pages**
- Up: **4,388 pages**
- New: **2,236 pages**
- Flat: **1,152 pages**

These statistics suggest that content performance varies considerably across pages, making content refresh prioritization a useful machine learning problem.

## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

# Careful Words

This project identifies patterns in historical data.

The model will not prove that refreshing a page causes better rankings. Many external factors, such as search engine algorithm updates and competitor actions, can also influence performance. The recommendations should therefore support, rather than replace, human decision-making.

- ✔ Selected a lane
- ✔ Defined the decision
- ✔ Defined the unit of analysis
- ✔ Defined the output
- ✔ Explained the action
- ✔ Explained the cost of a wrong recommendation
- ✔ Included real numbers from the dataset
- ✔ Explained why machine learning is appropriate